In [10]:
import duckdb
import pandas as pd
from pathlib import Path
import gc
import yaml
import logging
from collections import defaultdict, deque
from typing import Any
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True 
)


In [11]:
class ConnectionManager():
    def __init__(self, db_con_str):
        # la fonction duckdb.connect transforme le path en absolue, autant le faire ici 
        self.db_con_str = Path(db_con_str).expanduser().resolve()

        # connexion lazy
        self._con = None


    @property
    def con(self):
        if(self._con is None):
            # se connecter à la base de données
            self._con = duckdb.connect(self.db_con_str)

        return self._con


    @con.setter
    def con(self, value):
        # si au moment de changer la connexion on a déjà une connexion active
        if(self._con is not None):
            self._con.close() # cloturer la connexion en cours
            self._con = None # retirer la référence sur la connexion en cours
            gc.collect() # appeler le garbage collector pour forcer l'action de libérer les ressources et éviter les conflits d'accès

        self._con = value # pointer sur la nouvelle connexion
    
    
    def close_con(self):
        # on exploite le setter de la propriété pour cloturer correctement la connexion
        self.con = None


    def __del__(self):
        """Ferme automatiquement la connexion DuckDB quand l'objet est détruit."""
        try:
            self.close_con()
        except Exception:
            pass


    def __enter__(self):
        return self
    

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close_con()
        return False

In [12]:
class ConnectionUtils(ConnectionManager):
    def __init__(self, db_con_str : str):
        super().__init__(db_con_str)


    def tables(self):
        """Retourne la liste de toutes les tables physiques de la base courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            AND table_type = 'BASE TABLE'
            ORDER BY table_name
        """)


    def views(self):
        """Cette fonction renvoi la liste de toutes vues accessibles dans la base de données courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.views
            WHERE table_catalog = current_database()
            ORDER BY table_name
        """)


    def tables_views(self):
        """Retourne la liste des tables et des vues de la base courante"""
        return self.con.sql("""
            SELECT 
                table_name,
                table_type          -- 'BASE TABLE' ou 'VIEW'
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            ORDER BY table_type, table_name
        """)


    def table_exists(self, table_name : str):
        """Cette foction vérife qu'une table physique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{table_name}')
                AND
                (table_type = 'BASE TABLE')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]
    
    
    def view_exists(self, view_name : str):
        """Cette fonction check si une vue existe"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE 
                (table_name = '{view_name}')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def table_view_exists(self, name : str):
        """Cette foction vérife qu'une table physique ou une vue logique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{name}')
                AND
                (table_type = 'BASE TABLE' OR table_type = 'VIEW')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def drop_table_if_exists(self, table_name : str):
        """Cette fonction permet de supprimer une table s'elle existe"""
        self.con.sql(f"DROP TABLE IF EXISTS {table_name}")


    def drop_tables_if_exists(self, tables : list[str]):
        """Cette fonction surpprime chaque table de la liste tables s'elle existe dans la base courante"""
        for table_name in tables : 
            self.drop_table_if_exists(table_name)


    def drop_view_if_exists(self, view_name : str):
        """Cette fonction permet de supprimer une vue s'elle existe"""
        self.con.sql(f"DROP VIEW IF EXISTS {view_name}")


    def drop_views_if_exists(self, views : list[str]):
        """Cette fonction surpprime chaque vue de la liste views s'elle existe dans la base courante"""
        for view_name in views : 
            self.drop_view_if_exists(view_name)


    def table(self, table_name : str):
        """Cette fonction renvoi la table dont le nom est passé en paramètre"""
        return self.con.table(table_name)


    def view(self, view_name : str):
        """Cette fonction renvoi la vue dont le nom est passé en paramètre"""
        return self.con.view(view_name)


    def table_view(self, name : str):
        """Retourne la relation d'une table ou d'une vue selon ce qui existe."""
        return self.con.sql(f"SELECT * FROM {name}")


    def create_table_view_if_not_exists(self, name: str, sql: str, type: str = "VIEW"):
        """
        Crée une vue ou une table uniquement si elle n'existe pas encore.
        """
        sql = sql.strip().rstrip(";")
        self.con.sql(f"""CREATE {type} IF NOT EXISTS {name} AS ({sql})""")
        

In [13]:
class DependencyTree:
    def __init__(self, data: dict[str, dict[str, Any]]):
        """
        data : dictionnaire de la forme
        {
            "v_sales": {"requires": ["t_sales"], ...},
            "t_sales": {"requires": ["df_sales"], ...},
            ...
        }
        """
        self.data = data
        self.graph = self._build_graph()          # node -> list of dependencies
        self.reverse_graph = self._build_reverse_graph()  # node -> list of dependents

    def _build_graph(self) -> dict[str, list[str]]:
        return {
            name: config.get("requires", [])
            for name, config in self.data.items()
        }

    def _build_reverse_graph(self) -> dict[str, list[str]]:
        reverse = defaultdict(list)
        for node, deps in self.graph.items():
            for dep in deps:
                reverse[dep].append(node)
        return dict(reverse)

    # -------------------------------------------------------------------------
    # Informations de base
    # -------------------------------------------------------------------------
    def nodes(self) -> list[str]:
        """Retourne tous les nœuds du graphe."""
        return list(self.graph.keys())

    def dependencies(self, name: str) -> list[str]:
        """Retourne les dépendances directes d'un nœud."""
        return self.graph.get(name, [])

    def dependents(self, name: str) -> list[str]:
        """Retourne les nœuds qui dépendent directement de celui-ci."""
        return self.reverse_graph.get(name, [])

    def roots(self) -> list[str]:
        """Nœuds qui ne sont requis par personne."""
        all_deps = {dep for deps in self.graph.values() for dep in deps}
        return [n for n in self.graph if n not in all_deps]

    def leaves(self) -> list[str]:
        """Nœuds qui n'ont aucune dépendance."""
        return [n for n, deps in self.graph.items() if not deps]

    # -------------------------------------------------------------------------
    # Dépendances récursives
    # -------------------------------------------------------------------------
    def all_dependencies(self, name: str) -> list[str]:
        """Retourne toutes les dépendances (directes + indirectes) dans l'ordre topologique."""
        result = []
        visited = set()

        def dfs(node: str):
            if node in visited:
                return
            visited.add(node)
            for dep in self.graph.get(node, []):
                dfs(dep)
            result.append(node)

        dfs(name)
        return result[:-1]  # on retire le nœud lui-même

    def creation_order(self, name: str | None = None) -> list[str]:
        """
        Ordre de création (topologique).
        Si name est fourni → uniquement pour ce nœud et ses dépendances.
        Sinon → ordre global.
        """
        if name:
            nodes = self.all_dependencies(name) + [name]
        else:
            nodes = self.nodes()

        in_degree = {n: 0 for n in nodes}
        for n in nodes:
            for dep in self.graph.get(n, []):
                if dep in in_degree:
                    in_degree[n] += 1

        queue = deque([n for n, deg in in_degree.items() if deg == 0])
        order = []

        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent in self.reverse_graph.get(node, []):
                if dependent in in_degree:
                    in_degree[dependent] -= 1
                    if in_degree[dependent] == 0:
                        queue.append(dependent)

        return order

    # -------------------------------------------------------------------------
    # Affichage
    # -------------------------------------------------------------------------
    def print_tree(self, root: str | None = None):
        """Affiche l'arbre de dépendances en texte."""
        def _print(node: str, prefix: str = "", is_last: bool = True, visited: set | None = None):
            if visited is None:
                visited = set()

            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{node}")

            if node in visited:
                print(f"{prefix}{'    ' if is_last else '│   '}└── [cycle détecté]")
                return

            visited = visited | {node}
            deps = self.graph.get(node, [])
            new_prefix = prefix + ("    " if is_last else "│   ")

            for i, dep in enumerate(deps):
                _print(dep, new_prefix, i == len(deps) - 1, visited)

        if root:
            print(f"\n=== Dépendances de '{root}' ===\n")
            _print(root)
        else:
            print("\n=== Graphe complet ===\n")
            roots = self.roots()
            for i, r in enumerate(roots):
                _print(r, is_last=(i == len(roots) - 1))

    def print_levels(self, name: str | None = None):
        """Affiche les nœuds par niveau topologique."""
        order = self.creation_order(name)
        print(f"\n=== Ordre de création {'de ' + name if name else 'global'} ===\n")
        for i, node in enumerate(order, 1):
            print(f"{i:2d}. {node}")

In [14]:
class ConnectionPipeline(ConnectionUtils):
    def __init__(self, db_con_str : str, pipeline_file_path : str):
        super().__init__(db_con_str)
        self.pipeline_file_path = Path(pipeline_file_path).expanduser().resolve()
        self._pipeline = None
        self._tree = None


    def load_pipeline(self) -> dict:
        """Charge le fichier de définition des tables/vues."""
        with open(self.pipeline_file_path, "r", encoding="utf-8") as f:
            pipeline = yaml.safe_load(f)
            return pipeline


    @property
    def pipeline(self):
        if(self._pipeline is None):
            self._pipeline = self.load_pipeline()
        return self._pipeline


    @property
    def tree(self):
        if(self._tree is None):
            self._tree = DependencyTree(self.pipeline)
        return self._tree
    

    def df_from_file(self, file: str | Path, **kwargs) -> pd.DataFrame:
        """Charge un fichier en DataFrame selon son extension + options"""
        path = Path(file).expanduser().resolve()
        suffix = path.suffix.lower()

        if suffix in {".xlsx", ".xls", ".xlsm"}:
            return pd.read_excel(path, **kwargs)

        elif suffix == ".csv":
            return pd.read_csv(path, **kwargs)

        elif suffix == ".tsv":
            return pd.read_csv(path, sep="\t", **kwargs)

        elif suffix == ".json":
            return pd.read_json(path, **kwargs)

        elif suffix == ".parquet":
            return pd.read_parquet(path, **kwargs)

        else:
            raise ValueError(f"Extension non supportée : {suffix}")


    def df_from_file_config(self, config : dict):
        # On prépare les kwargs en enlevant les clés réservées
        reserved = {"type", "requires", "file"}
        kwargs = {k: v for k, v in config.items() if k not in reserved}

        return self.df_from_file(config["file"], **kwargs)


    def process_dataframe_type(self, name : str):
        if(name in self.pipeline):
            if(not self.table_view_exists(name)):
                config = self.pipeline[name]
                df = self.df_from_file_config(config)
                self.con.register(name, df)


    def process_table_view_type(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]
            self.create_table_view_if_not_exists(name, config["sql"], config["type"])


    def process(self, name : str):
        logging.getLogger().debug(f"process({name})")

        if(name in self.pipeline):
            config = self.pipeline[name]

            if(config["type"] == "dataframe"):
                self.process_dataframe_type(name)

            elif(config["type"] in ["table", "view"]):
                self.process_table_view_type(name)


    def process_with_requires(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]

            if(not self.table_view_exists(name)):
                for subname in config.get("requires", []):
                    self.process_with_requires(subname)

                self.process(name)


    def p_table_view(self, name : str):
        self.process_with_requires(name)
        return self.table_view(name)

In [15]:
class SalesPilBase(ConnectionPipeline):
    def __init__(self, db_con_str = "duckdb/pilotes/base/base.duckdb", pipeline_file_path = "config/base_pipeline.yaml"):
        super().__init__(db_con_str, pipeline_file_path)

    def sanitize(self, name: str) -> str:
        """Nettoie un nom pour en faire un préfixe de colonne valide."""
        name = str(name).strip().lower()
        name = re.sub(r"[^a-z0-9]+", "_", name)
        name = re.sub(r"_+", "_", name).strip("_")
        return name


    def pivot_sales_model_base(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Transforme v_sales_model_base (grain produit) en une table
        à **une ligne par hôtel et par scénario** avec colonnes préfixées :
        - produit__*
        - gamme__*
        - type__*
        - global__*

        Pour chaque ligne scénario/hôtel, **ajoute une 2e ligne** qui copie
        tout sauf ``metres_lineaires`` : ce champ devient la **somme exacte**
        des ``produit_metres_lineaires`` des produits du scénario
        (``mlin_source='sum_produits'``).
        """

        scenario_cols = [
            c for c in (
                "scenario_id",
                "scenario_label",
                "scenario_kind",
                "scenario_rank",
                "n_removed",
                "removed_items",
            )
            if c in df.columns
        ]
        id_cols = scenario_cols + [
            c for c in ("hotel_code", "hotel_name", "solution", "metres_lineaires")
            if c in df.columns
        ]

        product_metrics = [
            c for c in df.columns
            if c.startswith("produit_") and c not in id_cols
        ]
        gamme_metrics = [
            c for c in df.columns
            if c.startswith("gamme_") and c not in id_cols
        ]
        type_metrics = [
            c for c in df.columns
            if c.startswith("type_") and c not in id_cols
        ]
        global_metrics = [
            c for c in df.columns
            if not c.startswith(("produit_", "gamme_", "type_"))
            and c not in id_cols
            and c not in ["type", "gamme", "produit", "nombre_mois"]
            and c not in scenario_cols
        ]

        # S'assurer d'avoir un m_lin produit (fallback égalitaire)
        work = df.copy()
        if "produit_metres_lineaires" not in work.columns:
            if "nombre_metres_lineaires_par_produit" in work.columns:
                work["produit_metres_lineaires"] = pd.to_numeric(
                    work["nombre_metres_lineaires_par_produit"], errors="coerce"
                )
            elif "produit_part_des_produits" in work.columns and "metres_lineaires" in work.columns:
                work["produit_metres_lineaires"] = (
                    pd.to_numeric(work["metres_lineaires"], errors="coerce")
                    * pd.to_numeric(work["produit_part_des_produits"], errors="coerce")
                )
            else:
                work["produit_metres_lineaires"] = pd.NA
            if "produit_metres_lineaires" not in product_metrics:
                product_metrics = list(product_metrics) + ["produit_metres_lineaires"]
        if "produit_part_metres_lineaires" not in work.columns:
            if "produit_part_des_produits" in work.columns:
                work["produit_part_metres_lineaires"] = work["produit_part_des_produits"]
            if "produit_part_metres_lineaires" not in product_metrics:
                product_metrics = list(product_metrics) + ["produit_part_metres_lineaires"]

        if "scenario_id" in work.columns:
            group_keys = ["scenario_id", "hotel_code"]
        elif "scenario_label" in work.columns:
            group_keys = ["scenario_label", "hotel_code"]
        else:
            group_keys = ["hotel_code"]

        rows = []
        for _, g in work.groupby(group_keys, sort=False, dropna=False):
            row = {}
            first = g.iloc[0]
            for col in id_cols:
                row[col] = first[col]

            for _, r in g.iterrows():
                prefix = f"produit__{self.sanitize(r['produit'])}__"
                for m in product_metrics:
                    if m in r.index:
                        row[prefix + m] = r[m]

            subset_g = ["type", "gamme"] if "type" in g.columns else ["gamme"]
            gammes = g.drop_duplicates(subset=subset_g)
            for _, r in gammes.iterrows():
                prefix = f"gamme__{self.sanitize(r['gamme'])}__"
                for m in gamme_metrics:
                    if m in r.index:
                        row[prefix + m] = r[m]

            if "type" in g.columns:
                types = g.drop_duplicates(subset=["type"])
                for _, r in types.iterrows():
                    prefix = f"type__{self.sanitize(r['type'])}__"
                    for m in type_metrics:
                        if m in r.index:
                            row[prefix + m] = r[m]

            for m in global_metrics:
                row[f"global__{m}"] = first[m]

            # Ligne A : m_lin hôtel (tel que dans les données)
            row_hotel = dict(row)
            row_hotel["mlin_source"] = "hotel"
            rows.append(row_hotel)

            # Ligne B : copie exacte sauf metres_lineaires = Σ m_lin produits
            sum_prod_mlin = float(
                pd.to_numeric(g["produit_metres_lineaires"], errors="coerce").fillna(0).sum()
            )
            row_sum = dict(row)
            row_sum["metres_lineaires"] = sum_prod_mlin
            row_sum["mlin_source"] = "sum_produits"
            # distinguer le scénario pour ne pas collapser au rechargement
            if "scenario_id" in row_sum and row_sum["scenario_id"] is not None:
                base_sid = str(row_sum["scenario_id"])
                if not base_sid.endswith("__mlin_sum"):
                    row_sum["scenario_id"] = f"{base_sid}__mlin_sum"
            if "scenario_label" in row_sum and row_sum["scenario_label"] is not None:
                row_sum["scenario_label"] = f"{row_sum['scenario_label']} [m_lin=Σ produits]"
            rows.append(row_sum)

        result = pd.DataFrame(rows)
        result = result.fillna(0)
        return result


    def create_or_replace_sales_model_base_line_table(self, source: str = "v_sales_model_base"):
        """
        Pivot large → t_sales_model_base_line.
        Chaque (scénario × hôtel) produit **2 lignes** :
          - mlin_source=hotel
          - mlin_source=sum_produits (m_lin = somme des produit_metres_lineaires)
        """
        df = self.pivot_sales_model_base(self.p_table_view(source).df())
        self.con.register("v_sales_model_base_line", df)
        self.con.sql(
            "CREATE OR REPLACE TABLE t_sales_model_base_line AS "
            "SELECT * FROM v_sales_model_base_line"
        )
        return df


    @classmethod
    def main(cls):
        """Cette fonction représente la fonction principale qui exploite cette classe et qu'elle faut exécuter"""

        base = SalesPilBase()

        print(base.p_table_view("v_sales").df().shape)
        display(base.p_table_view("v_sales").df().head(3))

        print(base.p_table_view("v_sales_model").df().shape)
        display(base.p_table_view("v_sales_model").df().head(3))

        print(base.p_table_view("v_sales_model_base").df().shape)
        display(base.p_table_view("v_sales_model_base").df().head(3))

        line = base.create_or_replace_sales_model_base_line_table()
        print(line.shape, "# base : 2 lignes / hôtel (hotel m_lin + sum produits)")
        if "mlin_source" in line.columns:
            print(line.groupby("mlin_source").size())
        display(line.head(4))

        base.close_con()


In [16]:
SalesPilBase.main() # exécuter la fonction principale de la classe permettant de construire la base de données de modélisation

(130566, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F_B,NON-F&B,PAP,ACCESSOIRES,Tongs-Femme-100-Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F_B,NON-F&B,PAP,ACCESSOIRES,Casquette-Enfant-Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F_B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque-Easybreath-de-Surface-Adulte-500-Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(109342, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F_B,NON-F&B,PAP,ACCESSOIRES,Tongs-Femme-100-Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F_B,NON-F&B,PAP,ACCESSOIRES,Casquette-Enfant-Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F_B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque-Easybreath-de-Surface-Adulte-500-Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(848, 103)


,hotel_code,hotel_name,solution,metres_lineaires,type,gamme,produit,produit_nombre_ventes,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,nombre_mois,produit_nombre_ventes_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_achats,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_metres_lineaires_par_produit,nombre_produits_par_metre_lineaire,nombre_ventes_par_metre_lineaire,montant_ventes_par_metre_lineaire,montant_achats_par_metre_lineaire,montant_marge_par_metre_lineaire,produit_part_des_produits,nombre_ventes_par_mois,montant_ventes_par_mois,montant_achats_par_mois,montant_marge_par_mois,gamme_part_des_produits,type_part_des_produits,produit_part_nombre_ventes,produit_part_montant_ventes,produit_part_montant_achats,produit_part_montant_marge,gamme_part_nombre_ventes,gamme_part_montant_ventes,gamme_part_montant_achats,gamme_part_montant_marge,type_part_nombre_ventes,type_part_montant_ventes,type_part_montant_achats,type_part_montant_marge
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,F_B,ALCOOL,Bière-Duvel-33cl,14.0,56.0000,56.0,0.0000,4.000000,4.0,0.000000,24,0.583333,2.333333,2.333333,0.000000,1963.0,21,19795.4205,20475.0,-679.5795,10.084269,10.430464,-0.346194,93.47619,942.639071,975.0,-32.360929,81.791667,824.809187,853.125,-28.315813,15541.0,96,5,80937.8655,80458.0,479.8655,5.208022,5.177144,0.030877,161.885417,843.102766,838.104167,4.998599,3108.2,16187.5731,16091.6,95.9731,647.541667,3372.411062,3352.416667,19.994396,15759.0,142,9,2,84373.3057,83953.10225,420.20345,5.353976,5.327312,0.026664,110.978873,594.178209,591.21903,2.959179,1751.0,9374.811744,9328.122472,46.689272,7879.5,42186.65285,41976.551125,210.101725,0.042254,23.666667,2626.5,14062.217617,13992.183708,70.033908,0.007042,656.625,3515.554404,3498.045927,17.508477,0.147887,0.676056,0.000888,0.000664,0.000667,0.000000,0.124564,0.234617,0.243886,-1.617263,0.986167,0.959283,0.958368,1.141984
1,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,F_B,ALCOOL,Bière-Gallia-25cl,291.0,1991.0450,2037.0,-45.9550,6.842079,7.0,-0.157921,24,12.125000,82.960208,84.875000,-1.914792,1963.0,21,19795.4205,20475.0,-679.5795,10.084269,10.430464,-0.346194,93.47619,942.639071,975.0,-32.360929,81.791667,824.809187,853.125,-28.315813,15541.0,96,5,80937.8655,80458.0,479.8655,5.208022,5.177144,0.030877,161.885417,843.102766,838.104167,4.998599,3108.2,16187.5731,16091.6,95.9731,647.541667,3372.411062,3352.416667,19.994396,157

(14, 14720) # base : 2 lignes / hôtel (hotel m_lin + sum produits)
mlin_source
hotel           7
sum_produits    7
dtype: int64


,hotel_code,hotel_name,solution,metres_lineaires,produit__bi_re_duvel_33cl__produit_nombre_ventes,produit__bi_re_duvel_33cl__produit_montant_ventes,produit__bi_re_duvel_33cl__produit_montant_achats,produit__bi_re_duvel_33cl__produit_montant_marge,produit__bi_re_duvel_33cl__produit_montant_par_vente,produit__bi_re_duvel_33cl__produit_montant_achats_par_vente,produit__bi_re_duvel_33cl__produit_montant_marge_par_vente,produit__bi_re_duvel_33cl__produit_nombre_ventes_par_mois,produit__bi_re_duvel_33cl__produit_montant_ventes_par_mois,produit__bi_re_duvel_33cl__produit_montant_achats_par_mois,produit__bi_re_duvel_33cl__produit_montant_marge_par_mois,produit__bi_re_duvel_33cl__produit_part_des_produits,produit__bi_re_duvel_33cl__produit_part_nombre_ventes,produit__bi_re_duvel_33cl__produit_part_montant_ventes,produit__bi_re_duvel_33cl__produit_part_montant_achats,produit__bi_re_duvel_33cl__produit_part_montant_marge,produit__bi_re_duvel_33cl__produit_metres_lineaires,produit__bi_re_duvel_33cl__produit_part_metres_lineaires,produit__bi_re_gallia_25cl__produit_nombre_ventes,produit__bi_re_gallia_25cl__produit_montant_ventes,produit__bi_re_gallia_25cl__produit_montant_achats,produit__bi_re_gallia_25cl__produit_montant_marge,produit__bi_re_gallia_25cl__produit_montant_par_vente,produit__bi_re_gallia_25cl__produit_montant_achats_par_vente,produit__bi_re_gallia_25cl__produit_montant_marge_par_vente,produit__bi_re_gallia_25cl__produit_nombre_ventes_par_mois,produit__bi_re_gallia_25cl__produit_montant_ventes_par_mois,produit__bi_re_gallia_25cl__produit_montant_achats_par_mois,produit__bi_re_gallia_25cl__produit_montant_marge_par_mois,produit__bi_re_gallia_25cl__produit_part_des_produits,produit__bi_re_gallia_25cl__produit_part_nombre_ventes,produit__bi_re_gallia_25cl__produit_part_montant_ventes,produit__bi_re_gallia_25cl__produit_part_montant_achats,produit__bi_re_gallia_25cl__produit_part_montant_marge,produit__bi_re_gallia_25cl__produit_metres_lineaires,produit__bi_re_gallia_25cl__produit_part_metres_lineaires,produit__bi_re_heineken_25cl__produit_nombre_ventes,produit__bi_re_heineken_25cl__produit_montant_ventes,produit__bi_re_heineken_25cl__produit_montant_achats,produit__bi_re_heineken_25cl__produit_montant_marge,produit__bi_re_heineken_25cl__produit_montant_par_vente,produit__bi_re_heineken_25cl__produit_montant_achats_par_vente,produit__bi_re_heineken_25cl__produit_montant_marge_par_vente,produit__bi_re_heineken_25cl__produit_nombre_ventes_par_mois,produit__bi_re_heineken_25cl__produit_montant_ventes_par_mois,produit__bi_re_heineken_25cl__produit_montant_achats_par_mois,produit__bi_re_heineken_25cl__produit_montant_marge_par_mois,produit__bi_re_heineken_25cl__produit_part_des_produits,produit__bi_re_heineken_25cl__produit_part_nombre_ventes,produit__bi_re_heineken_25cl__produit_part_montant_ventes,produit__bi_re_heineken_25cl__produit_part_montant_achats,produit__bi_re_heineken_25cl__produit_part_montant_marge,produit__bi_re_heineken_25cl__produit_metres_lineaires,produit__bi_re_heineken_25cl__produit_part_metres_lineaires,produit__bi_re_session_p_le_ale_33cl__produit_nombre_ventes,produit__bi_re_session_p_le_ale_33cl__produit_montant_ventes,produit__bi_re_session_p_le_ale_33cl__produit_montant_achats,produit__bi_re_session_p_le_ale_33cl__produit_montant_marge,produit__bi_re_session_p_le_ale_33cl__produit_montant_par_vente,produit__bi_re_session_p_le_ale_33cl__produit_montant_achats_par_vente,produit__bi_re_session_p_le_ale_33cl__produit_montant_marge_par_vente,produit__bi_re_session_p_le_ale_33cl__produit_nombre_ventes_par_mois,produit__bi_re_session_p_le_ale_33cl__produit_montant_ventes_par_mois,produit__bi_re_session_p_le_ale_33cl__produit_montant_achats_par_mois,produit__bi_re_session_p_le_ale_33cl__produit_montant_marge_par_mois,produit__bi_re_session_p_le_ale_33cl__produit_part_des_produits,produit__bi_re_session_p_le_ale_33cl__produit_part_nombre_ventes,produit__bi_re_session_p_le_ale_33cl__produit_part_montant_ventes,produit

In [17]:
class SalesPilSim(SalesPilBase):
    """
    Simulations de retrait (produits / gammes / types).

    Chaque itération de retrait est un **scénario** :
      - insert dans ``t_sales_model_sim`` avec scenario_id / scenario_label
      - grain produit (plusieurs lignes / hôtel / scénario)
    À la fin, pivot → ``t_sales_model_base_line`` :
      **1 ligne par (hôtel × scénario)**.
    """
    def __init__(self, db_con_str = "duckdb/pilotes/sim/sim.duckdb", pipeline_file_path = "config/sim_pipeline.yaml"):
        super().__init__(db_con_str, pipeline_file_path)
        self._scenario_seq = 0


    def remove_elements(self, champs : str, elements : list[str]):
        """Recalcule les indicateurs après retrait d'éléments (sans INSERT)."""

        list_str = ",".join([f"'{elm}'" for elm in elements])

        self.p_table_view("t_sales_model")
        self.p_table_view("t_sales_model_base")

        self.drop_views_if_exists(self.views().df()["table_name"])

        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    COALESCE(SUM(CASE
                        WHEN {champs} IN ({list_str}) THEN QUANTITE
                        ELSE 0
                    END), 0) AS demande_quantite
                FROM
                    t_sales_model
                GROUP BY
                    hotel_code
                ORDER BY
                    hotel_code
            )
        """)

        self.con.sql("""
            CREATE OR REPLACE VIEW v_part_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    produit AS NOM_PRODUIT,
                    produit_nombre_ventes,
                    produit_part_nombre_ventes,
                    demande_quantite,
                    demande_quantite * produit_part_nombre_ventes AS part_demande_quantite
                FROM
                    v_demande_quantite
                LEFT JOIN
                    t_sales_model_base
                USING
                    (HOTEL_CODE)
                ORDER BY
                    HOTEL_CODE,
                    NOM_PRODUIT
            )
        """)

        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model_demande AS (
                SELECT 
                    SOLUTION,
                    HOTEL_CODE,
                    HOTEL_NAME,
                    METRES_LINEAIRES,
                    NOM_BOUTIQUE, 
                    TYPE,
                    TYPE_RAW,
                    GAMME,
                    GAMME_RAW,
                    NOM_PRODUIT,
                    NOM_PRODUIT_RAW,
                    CATEGORIE,
                    OPERATEUR,
                    MACHINE,
                    DATE,
                    HEURE,
                    STATUT,
                    CODE_EAN,
                    (QUANTITE / produit_nombre_ventes) * part_demande_quantite AS QUANTITE,
                    (PRIX_HT / produit_nombre_ventes)  * part_demande_quantite AS PRIX_HT,
                    VAT,
                    (PRIX_TTC / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC,
                    MARQUE,
                    FOURNISSEUR,
                    - ORDER_ID AS ORDER_ID,
                    TEMPERATURE,
                    (PRIX_TTC_MARCHE / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC_MARCHE,
                    (MARGE / produit_nombre_ventes)  * part_demande_quantite AS MARGE
                FROM 
                    t_sales_model
                RIGHT JOIN
                (
                    SELECT
                        *
                    FROM
                        v_part_demande_quantite
                    WHERE
                        part_demande_quantite > 0
                )
                USING
                    (HOTEL_CODE, NOM_PRODUIT)
            )
            """)

        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_sales_model AS (
                SELECT
                    *
                FROM
                    t_sales_model
                WHERE
                    {champs} NOT IN ({list_str})

                UNION ALL

                SELECT
                    *
                FROM
                    v_sales_model_demande
                WHERE
                    {champs} NOT IN ({list_str})
            )
        """)

        # recalcule indicateurs (pipeline) à partir de v_sales_model modifié
        self.p_table_view("v_sales_model_base")


    def remove_produits(self, produits : list[str]):
        self.remove_elements("NOM_PRODUIT", produits)

    def remove_gammes(self, gammes : list[str]):
        self.remove_elements("GAMME", gammes)

    def remove_types(self, types : list[str]):
        self.remove_elements("TYPE", types)


    def _ensure_sim_table_with_scenario_cols(self):
        """
        t_sales_model_sim = base (scénario baseline) + colonnes scénario.
        Recrée la table si les colonnes scénario manquent (anciennes runs).
        """
        self.p_table_view("t_sales_model_base")
        self._ensure_base_product_mlin_columns()
        need_rebuild = True
        if self.table_exists("t_sales_model_sim"):
            cols = [
                r[0]
                for r in self.con.sql("DESCRIBE t_sales_model_sim").fetchall()
            ]
            if "scenario_id" in cols and "scenario_label" in cols:
                need_rebuild = False
        if need_rebuild:
            self.con.sql("DROP TABLE IF EXISTS t_sales_model_sim")
            self.con.sql("""
                CREATE TABLE t_sales_model_sim AS
                SELECT
                    'base' AS scenario_id,
                    'base (aucun retrait)' AS scenario_label,
                    'base' AS scenario_kind,
                    0 AS scenario_rank,
                    0 AS n_removed,
                    CAST('' AS VARCHAR) AS removed_items,
                    *
                FROM t_sales_model_base
            """)
            self._scenario_seq = 0
            logging.debug("t_sales_model_sim recréée avec colonnes scénario + baseline")
        else:
            # reprendre le max rank
            try:
                mx = self.con.sql(
                    "SELECT COALESCE(MAX(scenario_rank), 0) FROM t_sales_model_sim"
                ).fetchone()[0]
                self._scenario_seq = int(mx or 0)
            except Exception:
                self._scenario_seq = 0


    def _next_scenario_meta(
        self,
        kind: str,
        elements: list[str],
    ) -> dict:
        self._scenario_seq += 1
        rank = self._scenario_seq
        n = len(elements)
        # label court : kind + n retirés + aperçu
        preview = ", ".join(str(e) for e in elements[:3])
        if n > 3:
            preview += f", … (+{n - 3})"
        sid = f"{kind}_{rank:04d}_n{n}"
        label = f"{kind}: retrait de {n} — {preview}" if n else f"{kind}: baseline"
        return {
            "scenario_id": sid,
            "scenario_label": label,
            "scenario_kind": kind,
            "scenario_rank": rank,
            "n_removed": n,
            "removed_items": " | ".join(str(e) for e in elements),
        }


    def _table_columns(self, name: str) -> list[str]:
        return [r[0] for r in self.con.sql(f"DESCRIBE {name}").fetchall()]


    def _ensure_base_product_mlin_columns(self):
        """
        Figé l'allocation m_lin produit sur l'assortiment **de base**.

        produit_metres_lineaires = metres_lineaires_hotel × (1/N_base)
        (ou part déjà présente). Ne doit PAS être recalculé après un retrait
        sinon la somme des restants re-remplit toujours le corner entier.
        """
        self.p_table_view("t_sales_model_base")
        cols = self._table_columns("t_sales_model_base")
        if "produit_metres_lineaires" not in cols:
            # ajoute les colonnes sans perdre le reste
            self.con.sql("""
                CREATE OR REPLACE TABLE t_sales_model_base AS
                SELECT
                    t.*,
                    t.produit_part_des_produits AS produit_part_metres_lineaires,
                    (t.metres_lineaires * t.produit_part_des_produits) AS produit_metres_lineaires
                FROM t_sales_model_base t
            """)
            logging.debug("t_sales_model_base : produit_metres_lineaires ajouté (figé base)")
        else:
            # recalcule si tout est null
            n_null = self.con.sql(
                "SELECT COUNT(*) FROM t_sales_model_base "
                "WHERE produit_metres_lineaires IS NULL"
            ).fetchone()[0]
            if n_null:
                self.con.sql("""
                    UPDATE t_sales_model_base
                    SET
                        produit_part_metres_lineaires = COALESCE(
                            produit_part_metres_lineaires, produit_part_des_produits
                        ),
                        produit_metres_lineaires = COALESCE(
                            produit_metres_lineaires,
                            metres_lineaires * produit_part_des_produits
                        )
                """)


    def _scenario_snap_with_frozen_product_mlin(self) -> str:
        """
        Snapshot indicateurs du scénario courant, avec m_lin **produit figé base**.

        Les produits restants gardent leur part de m_lin d'origine →
        Σ produit_metres_lineaires < metres_lineaires hôtel quand on retire
        des produits (vrai corner « exposé » plus petit).
        """
        self.p_table_view("v_sales_model_base")
        self._ensure_base_product_mlin_columns()
        cur_cols = self._table_columns("v_sales_model_base")
        exclude = [
            c for c in ("produit_metres_lineaires", "produit_part_metres_lineaires")
            if c in cur_cols
        ]
        excl_sql = f" EXCLUDE ({', '.join(exclude)})" if exclude else ""
        # Recolle le m_lin figé base sans changer l'ordre des colonnes (UPDATE via join)
        self.con.sql("CREATE OR REPLACE TEMP TABLE _scenario_snap AS SELECT * FROM v_sales_model_base")
        # colonnes absentes → les ajouter
        snap_cols = self._table_columns("_scenario_snap")
        if "produit_metres_lineaires" not in snap_cols:
            self.con.sql(
                "ALTER TABLE _scenario_snap ADD COLUMN produit_metres_lineaires DOUBLE"
            )
        if "produit_part_metres_lineaires" not in snap_cols:
            self.con.sql(
                "ALTER TABLE _scenario_snap ADD COLUMN produit_part_metres_lineaires DOUBLE"
            )
        self.con.sql("""
            UPDATE _scenario_snap AS cur
            SET
                produit_metres_lineaires = COALESCE(b.produit_metres_lineaires, 0),
                produit_part_metres_lineaires = COALESCE(
                    b.produit_part_metres_lineaires,
                    b.produit_part_des_produits,
                    0
                )
            FROM t_sales_model_base AS b
            WHERE cur.hotel_code = b.hotel_code
              AND cur.produit = b.produit
        """)
        # produits absents de la base → 0
        self.con.sql("""
            UPDATE _scenario_snap
            SET
                produit_metres_lineaires = COALESCE(produit_metres_lineaires, 0),
                produit_part_metres_lineaires = COALESCE(produit_part_metres_lineaires, 0)
        """)
        # excl_sql unused but kept for signature clarity
        _ = excl_sql
        return "_scenario_snap"


    def _insert_current_base_as_scenario(self, meta: dict):
        """INSERT le snapshot scénario (m_lin produit figé base) tagué."""
        def esc(v):
            return str(v).replace("'", "''")

        snap = self._scenario_snap_with_frozen_product_mlin()

        # BY NAME : évite le décalage si produit_metres_lineaires est recollé en fin de SELECT
        self.con.sql(f"""
            INSERT INTO t_sales_model_sim BY NAME
            SELECT
                '{esc(meta["scenario_id"])}' AS scenario_id,
                '{esc(meta["scenario_label"])}' AS scenario_label,
                '{esc(meta["scenario_kind"])}' AS scenario_kind,
                {int(meta["scenario_rank"])} AS scenario_rank,
                {int(meta["n_removed"])} AS n_removed,
                '{esc(meta["removed_items"])}' AS removed_items,
                snap.*
            FROM {snap} AS snap
        """)
        n = self.con.sql(
            f"SELECT COUNT(*) FROM t_sales_model_sim WHERE scenario_id = '{esc(meta['scenario_id'])}'"
        ).fetchone()[0]
        # diagnostic : m_lin exposé (Σ produits) vs m_lin hôtel
        try:
            diag = self.con.sql(f"""
                SELECT
                    hotel_code,
                    max(metres_lineaires) AS m_hotel,
                    sum(produit_metres_lineaires) AS m_expose,
                    count(*) AS n_prod
                FROM t_sales_model_sim
                WHERE scenario_id = '{esc(meta["scenario_id"])}'
                GROUP BY hotel_code
                ORDER BY hotel_code
                LIMIT 3
            """).df()
            logging.debug(
                f"INSERT scenario {meta['scenario_id']} → {n} lignes produit "
                f"(rank={meta['scenario_rank']}, n_removed={meta['n_removed']})\n"
                f"m_lin exposé (extrait):\n{diag.to_string(index=False)}"
            )
        except Exception as exc:
            logging.debug(
                f"INSERT scenario {meta['scenario_id']} → {n} lignes "
                f"(diag m_lin skip: {exc})"
            )


    def multiple_remove_elements(self, champs : str, elements : list[str], kind: str | None = None):
        """
        Pour i = 1..len(elements) : retire les i premiers éléments, calcule
        les indicateurs, INSERT dans t_sales_model_sim avec un scenario_id.
        """
        kind = kind or str(champs).lower()
        self._ensure_sim_table_with_scenario_cols()

        elements = list(elements)
        # i=1 .. len(elements) inclus : retrait cumulatif de 1, 2, …, n éléments
        for i in range(1, len(elements) + 1):
            sub_elements = elements[:i]
            logging.debug(
                f"champs={champs} scenario {i}/{len(elements)} "
                f"removed={len(sub_elements)}"
            )
            self.remove_elements(champs, sub_elements)
            meta = self._next_scenario_meta(kind, sub_elements)
            self._insert_current_base_as_scenario(meta)


    def multiple_remove_produits(self, produits):
        self.multiple_remove_elements("NOM_PRODUIT", list(produits), kind="produit")

    def multiple_remove_gammes(self, gammes):
        self.multiple_remove_elements("GAMME", list(gammes), kind="gamme")

    def multiple_remove_types(self, types):
        self.multiple_remove_elements("TYPE", list(types), kind="type")


    def multiple_remove_produits_gammes_types(self):
        self.p_table_view("v_produits")
        self.p_table_view("v_gammes")
        self.p_table_view("v_types")

        # listes distinctes globales (ordre stable)
        produits = (
            self.con.sql("SELECT DISTINCT produit FROM v_produits ORDER BY produit")
            .df()["produit"]
            .tolist()
        )
        gammes = (
            self.con.sql("SELECT DISTINCT gamme FROM v_gammes ORDER BY gamme")
            .df()["gamme"]
            .tolist()
        )
        types = (
            self.con.sql("SELECT DISTINCT type FROM v_types ORDER BY type")
            .df()["type"]
            .tolist()
        )

        logging.info(
            f"Scénarios prévus : "
            f"produits={len(produits)} + gammes={len(gammes)} + types={len(types)} "
            f"+ 1 baseline → "
            f"~{(len(produits)+len(gammes)+len(types)+1)*7} lignes hôtel×scénario "
            f"(si 7 hôtels)"
        )

        # Table sim = baseline taguée
        self.con.sql("DROP TABLE IF EXISTS t_sales_model_sim")
        self._ensure_sim_table_with_scenario_cols()

        logging.debug("multiple_remove_produits")
        self.multiple_remove_produits(produits)

        logging.debug("multiple_remove_gammes")
        self.multiple_remove_gammes(gammes)

        logging.debug("multiple_remove_types")
        self.multiple_remove_types(types)

        # Vue multi-scénarios pour le pivot
        self.con.sql(
            "CREATE OR REPLACE VIEW v_sales_model_base AS "
            "SELECT * FROM t_sales_model_sim"
        )

        line = self.create_or_replace_sales_model_base_line_table(
            source="t_sales_model_sim"
        )
        n_scen = self.con.sql(
            "SELECT COUNT(DISTINCT scenario_id) FROM t_sales_model_sim"
        ).fetchone()[0]
        n_hotels = self.con.sql(
            "SELECT COUNT(DISTINCT hotel_code) FROM t_sales_model_sim"
        ).fetchone()[0]
        logging.info(
            f"t_sales_model_base_line = {line.shape[0]} lignes × {line.shape[1]} cols "
            f"({n_scen} scénarios × {n_hotels} hôtels attendu ≈ {n_scen * n_hotels})"
        )
        return line


    @classmethod
    def main(cls):
        """Pipeline complet base + simulations multi-scénarios + pivot large."""
        super().main()

        sim = SalesPilSim()
        line = sim.multiple_remove_produits_gammes_types()
        print("line shape (hôtel × scénario) :", line.shape)
        if "scenario_id" in line.columns:
            print(
                "n_scenarios:",
                line["scenario_id"].nunique(),
                "n_hotels:",
                line["hotel_code"].nunique(),
            )
            display(line[["scenario_id", "scenario_label", "hotel_code"]].head(20))
        else:
            display(line.head(3))
        sim.close_con()


In [ ]:
SalesPilSim().main()

In [45]:
df_h = pd.read_excel("data/hotel_data.xlsx")
df_h.shape

(5723, 64)

In [57]:
df = pd.read_excel("data/hotel_sales_raw_extended_data.xlsx")

In [58]:
print(df.shape)
df.head(3)

(130566, 34)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,HOTEL_NB_CHAMBRES,HOTEL_TO_ANNUEL,NOM_BOUTIQUE,TYPE_RAW,TYPE,GAMME_RAW,GAMME,NOM_PRODUIT_RAW,NOM_PRODUIT,NATURE_PRODUIT,MACHINE_RAW,MACHINE,MARQUE_RAW,MARQUE,FOURNISSEUR_RAW,FOURNISSEUR,CATEGORIE,OPERATEUR,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,129.0,0.72,Ibis budget Nice,NON-F&B,NON_F_B,ACCESSOIRES,ACCESSOIRES,TONGS FEMME 100 NOIR,TONGS FEMME 100 NOIR,TONGS,BORNE NICE,BORNE,DECATHLON,DECATHLON,DECATHLON,DECATHLON,NON_F_B,ADIPOS,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,9281,25.5,3.0,3.0
1,simply,H2075,Ibis budget Nice Californie,6.0,129.0,0.72,Ibis budget Nice,NON-F&B,NON_F_B,ACCESSOIRES,ACCESSOIRES,CASQUETTE ENFANT -MH100,CASQUETTE ENFANT MH100,CASQUETTE,BORNE NICE,BORNE,DECATHLON,DECATHLON,DECATHLON,DECATHLON,NON_F_B,ADIPOS,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,9282,25.5,8.0,4.0
2,simply,H2075,Ibis budget Nice Californie,6.0,129.0,0.72,Ibis budget Nice,NON-F&B,NON_F_B,ACCESSOIRES,ACCESSOIRES,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,MASQUE EASYBREATH DE SURFACE ADULTE 500 BLEU,MASQUE,BORNE NICE,BORNE,DECATHLON,DECATHLON,DECATHLON,DECATHLON,NON_F_B,ADIPOS,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,9283,25.5,7.0,22.0


In [61]:
print(df.shape)
df.head(3)

(130566, 34)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,HOTEL_NB_CHAMBRES,HOTEL_TO_ANNUEL,NOM_BOUTIQUE,TYPE_RAW,TYPE,GAMME_RAW,GAMME,NOM_PRODUIT_RAW,NOM_PRODUIT,NATURE_PRODUIT,MACHINE_RAW,MACHINE,MARQUE_RAW,MARQUE,FOURNISSEUR_RAW,FOURNISSEUR,CATEGORIE,OPERATEUR,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,129.0,0.72,Ibis budget Nice,NON-F&B,NON_F_B,ACCESSOIRES,ACCESSOIRES,TONGS FEMME 100 NOIR,TONGS FEMME 100 NOIR,TONGS,BORNE NICE,BORNE,DECATHLON,DECATHLON,DECATHLON,DECATHLON,NON_F_B,ADIPOS,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,9281,25.5,3.0,3.0
1,simply,H2075,Ibis budget Nice Californie,6.0,129.0,0.72,Ibis budget Nice,NON-F&B,NON_F_B,ACCESSOIRES,ACCESSOIRES,CASQUETTE ENFANT -MH100,CASQUETTE ENFANT MH100,CASQUETTE,BORNE NICE,BORNE,DECATHLON,DECATHLON,DECATHLON,DECATHLON,NON_F_B,ADIPOS,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,9282,25.5,8.0,4.0
2,simply,H2075,Ibis budget Nice Californie,6.0,129.0,0.72,Ibis budget Nice,NON-F&B,NON_F_B,ACCESSOIRES,ACCESSOIRES,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,MASQUE EASYBREATH DE SURFACE ADULTE 500 BLEU,MASQUE,BORNE NICE,BORNE,DECATHLON,DECATHLON,DECATHLON,DECATHLON,NON_F_B,ADIPOS,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,9283,25.5,7.0,22.0


In [53]:
df_h.head()

,hotel_code,hotel_name,hotel_brand,hotel_solution_simply,hotel_solution_liberty,hotel_solution_connected,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_city,hotel_country,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_has_parking,hotel_has_wifi,hotel_has_clim,hotel_has_petit_dejeuner,hotel_has_accessible,hotel_has_animaux,hotel_has_non_fumeur,hotel_has_navette,hotel_has_reunion,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner,hotel_contrat_type_franchise,hotel_contrat_type_manage
0,HB504,21c Museum Hotel Louisville,21C MUSEUM HOTELS,0,0,0,700 West Main Street,NaN,40202,LOUISVILLE,US,38.256782,-85.761776,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,1,1,0,0,1.0,0,0,1,1,0,1.0,0.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HB505,21c Museum Hotel Bentonville,21C MUSEUM HOTELS,0,0,0,200 NE A Street,NaN,72712,BENTONVILLE,US,36.374144,-94.207790,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,1,1,0,0,1.0,0,1,1,1,1,1.0,1.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HB507,21c Museum Hotel Cincinnati,21C MUSEUM HOTELS,0,0,0,609 Walnut Street,NaN,45202,CINCINNATI,US,39.103011,-84.512019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,1,1,0,0,1.0,0,0,1,1,0,1.0,0.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,HB508,21C Museum Hotel Durham,21C MUSEUM HOTELS,0,0,0,111 North Corcoran Street,NaN,27701,DURHAM,US,35.996060,-78.901787,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,1,1,0,0,1.0,0,1,1,1,0,1.0,0.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HB510,21C Museum Hotel Lexington,21C MUSEUM HOTELS,0,0,0,167 West Main Street,NaN,40507,LEXINGTON,US,38.047194,-84.497509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,1,1,0,0,1.0,0,1,1,1,1,1.0,0.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
df[df["MARGE"].isnull()][["NATURE_PRODUIT", "PRIX_TTC", "PRIX_TTC_MARCHE", "MARGE"]].drop_duplicates()

,NATURE_PRODUIT,PRIX_TTC,PRIX_TTC_MARCHE,MARGE


In [64]:
df[["HOTEL_CODE", "MARQUE", "FOURNISSEUR"]].drop_duplicates()

df[[ "MARQUE"]].sort_values(by = ["MARQUE"]).drop_duplicates()
# df[[ "FOURNISSEUR"]].sort_values(by = ["FOURNISSEUR"]).drop_duplicates()
# df[[ "CATEGORIE"]].sort_values(by = ["CATEGORIE"]).drop_duplicates()

,MARQUE
55277,ALAIN MILLIAT
35615,BACCHANTE
69729,BADOIT
36266,BAHIA
5533,BAM&CO
...,...
23642,TUC
125527,TWIX
62300,VITAO
25110,VITTEL


In [40]:
df[[ "NATURE_PRODUIT"]].sort_values(by = ["NATURE_PRODUIT"]).drop_duplicates()

,NATURE_PRODUIT
37345,Adaptateur
22502,Adaptateur Click USB C USB
35544,Adaptateur Europe > Monde
38752,Adaptateur Novotel
105787,Adaptateur Universel
...,...
49297,Yaourt Fraise Framboise
124667,Yaourt Nature
31177,de Champagne
34641,de Vin


In [50]:
df_h[df_h["hotel_nb_chambres"] > 0][["hotel_code", "hotel_nb_chambres", "hotel_to_annuel"]].drop_duplicates()

,hotel_code,hotel_nb_chambres,hotel_to_annuel
1828,H2075,129.0,NaN
2366,HB6A3,97.0,0.70
2437,H0815,309.0,0.95
3258,H0373,305.0,NaN
3547,H6188,191.0,NaN
4719,H3546,764.0,NaN
4987,HB5I0,572.0,NaN
